![Two data scientists working on a dashboard.](hr-image-small.png)

A common problem when creating models to generate business value from data is that the datasets can be so large that it can take days for the model to generate predictions. Ensuring that your dataset is stored as efficiently as possible is crucial for allowing these models to run on a more reasonable timescale without having to reduce the size of the dataset.

You've been hired by a major online data science training provider called *Training Data Ltd.* to clean up one of their largest customer datasets. This dataset will eventually be used to predict whether their students are looking for a new job or not, information that they will then use to direct them to prospective recruiters.

You've been given access to `customer_train.csv`, which is a subset of their entire customer dataset, so you can create a proof-of-concept of a much more efficient storage solution. The dataset contains anonymized student information, and whether they were looking for a new job or not during training:

| Column                   | Description                                                                      |
|------------------------- |--------------------------------------------------------------------------------- |
| `student_id`             | A unique ID for each student.                                                    |
| `city`                   | A code for the city the student lives in.                                        |
| `city_development_index` | A scaled development index for the city.                                         |
| `gender`                 | The student's gender.                                                            |
| `relevant_experience`    | An indicator of the student's work relevant experience.                          |
| `enrolled_university`    | The type of university course enrolled in (if any).                              |
| `education_level`        | The student's education level.                                                   |
| `major_discipline`       | The educational discipline of the student.                                       |
| `experience`             | The student's total work experience (in years).                                  |
| `company_size`           | The number of employees at the student's current employer.                       |
| `company_type`           | The type of company employing the student.                                       |
| `last_new_job`           | The number of years between the student's current and previous jobs.             |
| `training_hours`         | The number of hours of training completed.                                       |
| `job_change`             | An indicator of whether the student is looking for a new job (`1`) or not (`0`). |

In [1]:
# ============================================================
# Customer Analytics: Preparing Data for Modeling
# ============================================================

import pandas as pd
from pandas.api.types import CategoricalDtype

# Load the dataset
ds_jobs = pd.read_csv("customer_train.csv")

# Create a copy for transformation
ds_jobs_transformed = ds_jobs.copy()
print(ds_jobs.shape)

(19158, 14)


In [2]:
ds_jobs_transformed.head(3)

,student_id,city,city_development_index,gender,relevant_experience,enrolled_university,education_level,major_discipline,experience,company_size,company_type,last_new_job,training_hours,job_change
0,8949,city_103,0.920,Male,Has relevant experience,no_enrollment,Graduate,STEM,>20,NaN,NaN,1,36,1.0
1,29725,city_40,0.776,Male,No relevant experience,no_enrollment,Graduate,STEM,15,50-99,Pvt Ltd,>4,47,0.0
2,11561,city_21,0.624,NaN,No relevant experience,Full time course,Graduate,STEM,5,NaN,NaN,never,83,0.0


In [3]:
ds_jobs.apply(lambda x: x.unique())

student_id                [8949, 29725, 11561, 33241, 666, 21651, 28806,...
city                      [city_103, city_40, city_21, city_115, city_16...
city_development_index    [0.92, 0.7759999999999999, 0.624, 0.789, 0.767...
gender                                           [Male, nan, Female, Other]
relevant_experience       [Has relevant experience, No relevant experience]
enrolled_university       [no_enrollment, Full time course, nan, Part ti...
education_level           [Graduate, Masters, High School, nan, Phd, Pri...
major_discipline          [STEM, Business Degree, nan, Arts, Humanities,...
experience                [>20, 15, 5, <1, 11, 13, 7, 17, 2, 16, 1, 4, 1...
company_size              [nan, 50-99, <10, 10000+, 5000-9999, 1000-4999...
company_type              [nan, Pvt Ltd, Funded Startup, Early Stage Sta...
last_new_job                                   [1, >4, never, 4, 3, 2, nan]
training_hours            [36, 47, 83, 52, 8, 24, 18, 46, 123, 32, 108, ...
job_change  

In [4]:
# ------------------------------------------------------------
# 1. Binary categorical columns -> bool
# ------------------------------------------------------------
ds_jobs_transformed["relevant_experience"] = (
    ds_jobs_transformed["relevant_experience"]
    .map({
        "Has relevant experience": True,
        "No relevant experience": False
    })
    .astype("bool")
)

ds_jobs_transformed["job_change"] = (
    ds_jobs_transformed["job_change"]
    .map({1.0: True, 0.0: False})
    .astype("bool")
)


In [5]:
# ------------------------------------------------------------
# 2. Integer columns -> int32
# ------------------------------------------------------------
ds_jobs_transformed = ds_jobs_transformed.astype({
    "student_id": "int32",
    "training_hours": "int32",
})

# ------------------------------------------------------------
# 3. Float columns -> float16
# ------------------------------------------------------------
ds_jobs_transformed = ds_jobs_transformed.astype({
    "city_development_index": "float16",
})

In [6]:
# ------------------------------------------------------------
# 4. Nominal categorical columns -> category
# ------------------------------------------------------------
nominal_cols = ["city", "gender", "major_discipline", "company_type"]

for col in nominal_cols:
    ds_jobs_transformed[col] = ds_jobs_transformed[col].astype("category")


In [7]:
# ------------------------------------------------------------
# 5. Ordinal categorical columns -> ordered category
# ------------------------------------------------------------
education_level_order = [
    "Primary School", "High School", "Graduate", "Masters", "Phd"
]

# Includes "1", which was missing in the original notebook
experience_order = ["<1"] + [str(i) for i in range(1, 21)] + [">20"]

company_size_order = [
    "<10", "10-49", "50-99", "100-499",
    "500-999", "1000-4999", "5000-9999", "10000+"
]

last_new_job_order = ["never", "1", "2", "3", "4", ">4"]

enrolled_university_order = [
    "no_enrollment", "Part time course", "Full time course"
]

ordinal_cols = {
    "education_level": education_level_order,
    "experience": experience_order,
    "company_size": company_size_order,
    "last_new_job": last_new_job_order,
    "enrolled_university": enrolled_university_order,
}

for col, order in ordinal_cols.items():
    ds_jobs_transformed[col] = ds_jobs_transformed[col].astype(
        CategoricalDtype(categories=order, ordered=True)
    )

In [8]:

# ------------------------------------------------------------
# 6. Filter: 10+ years experience at companies with 1000+ employees
# ------------------------------------------------------------
ds_jobs_transformed = ds_jobs_transformed[
    (ds_jobs_transformed["experience"] >= "10") &
    (ds_jobs_transformed["company_size"] >= "1000-4999")
].copy()

# ------------------------------------------------------------
# 7. Validation
# ------------------------------------------------------------
print(ds_jobs_transformed.dtypes)
print(f"Shape: {ds_jobs_transformed.shape}")

original_memory = ds_jobs.memory_usage(deep=True).sum() / 1024**2
transformed_memory = ds_jobs_transformed.memory_usage(deep=True).sum() / 1024**2

print(f"Original memory: {original_memory:.2f} MB")
print(f"Transformed memory: {transformed_memory:.2f} MB")

student_id                   int32
city                      category
city_development_index     float16
gender                    category
relevant_experience           bool
enrolled_university       category
education_level           category
major_discipline          category
experience                category
company_size              category
company_type              category
last_new_job              category
training_hours               int32
job_change                    bool
dtype: object
Shape: (2201, 14)
Original memory: 11.82 MB
Transformed memory: 0.08 MB
